In [2]:
pip install rouge-score sacrebleu evaluate torchsummary

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import necessary libraries
import torch  # PyTorch for deep learning
import numpy as np  # NumPy for numerical operations
import pandas as pd  # Pandas for data manipulation
import re  # Regular expressions for text processing
import tensorflow as tf  # TensorFlow (not used in this script, but imported)
import evaluate  # Library for evaluation metrics
import seaborn as sns  # Seaborn for data visualization
import matplotlib.pyplot as plt  # Matplotlib for plotting
import warnings  # To suppress warnings

# Import Hugging Face Transformers components
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq, TrainerCallback, T5Config

# Import datasets for handling data
from datasets import Dataset

# Import scikit-learn for train-test split
from sklearn.model_selection import train_test_split

# Import PyTorch components for loss, optimization, and data handling
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from torch.utils.data import TensorDataset
from torchsummary import summary

# Import defaultdict for handling dictionaries with default values
from collections import defaultdict

# Suppress warnings to keep the output clean
warnings.filterwarnings("ignore")

2025-11-28 19:41:20.536651: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764358880.735322      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764358880.786490      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [4]:
# Load the dataset from a CSV file
df = pd.read_csv('/kaggle/input/layoutlm/medquad.csv')

# Display a sample of the data to understand its structure
print("Data Sample:")
print(df.head())

# Check for null values in the dataset
print("Null Value Data:")
print(df.isnull().sum())

# Define a list of common question words to filter relevant questions
question_words = ['what', 'who', 'why', 'when', 'where', 'how', 'is', 'are', 'does', 'do', 'can', 'will', 'shall']

# Convert all questions to lowercase for consistent filtering
df['question'] = df['question'].str.lower()

# Filter rows where the question starts with one of the question words
df = df[df['question'].str.split().str[0].isin(question_words)]

# Reset the index after filtering
df = df.reset_index(drop=True)

# Check for duplicate rows in the dataset
duplicates = df.duplicated()
print(f"Number of duplicate rows: {duplicates.sum()}")

# Remove duplicate rows to ensure data uniqueness
df = df.drop_duplicates()

# Reset the index after removing duplicates
df.reset_index(drop=True, inplace=True)

# Drop unused columns ('source' and 'focus_area') to simplify the dataset
df = df.drop(columns=['source', 'focus_area'])

# Display dataset information (columns, data types, and non-null counts)
print("Table Info:")
print(df.info())

# Remove duplicate rows based on the 'question' and 'answer' columns
df = df.drop_duplicates(subset='question', keep='first').reset_index(drop=True)
df = df.drop_duplicates(subset='answer', keep='first').reset_index(drop=True)

# Drop rows with null values in the 'question' or 'answer' columns
df = df.dropna(subset=['question', 'answer']).reset_index(drop=True)

# Fill any remaining null values with empty strings and convert to string type
df['question'] = df['question'].fillna('').astype(str)
df['answer'] = df['answer'].fillna('').astype(str)

# Define a function to clean text by removing parentheses and extra spaces
def clean_text(text):
    text = re.sub(r"\(.*?\)", "", text)  # Remove text within parentheses
    text = re.sub(r'\s+', ' ', text.strip().lower())  # Normalize spaces and convert to lowercase
    return text

# Apply the clean_text function to the 'question' and 'answer' columns
df['question'] = df['question'].apply(clean_text)
df['answer'] = df['answer'].apply(clean_text)

# Further clean the text by ensuring lowercase, stripping whitespace, and normalizing spaces
df['question'] = df['question'].str.lower().str.strip().apply(lambda x: re.sub(r'\s+', ' ', x))
df['answer'] = df['answer'].str.lower().str.strip().apply(lambda x: re.sub(r'\s+', ' ', x))

# Check for null values again after cleaning
print("Null Value Data After Cleaning:")
print(df.isnull().sum())

# Check the number of unique questions and answers in the dataset
print(f"Unique questions: {df['question'].nunique()}")
print(f"Unique answers: {df['answer'].nunique()}")

# Display dataset information and a sample of the cleaned data
print("Final Dataset Info:")
df.info()
print("Final Data Sample:")
df.head()

Data Sample:
                                 question  \
0                What is (are) Glaucoma ?   
1                  What causes Glaucoma ?   
2     What are the symptoms of Glaucoma ?   
3  What are the treatments for Glaucoma ?   
4                What is (are) Glaucoma ?   

                                              answer           source  \
0  Glaucoma is a group of diseases that can damag...  NIHSeniorHealth   
1  Nearly 2.7 million people have glaucoma, a lea...  NIHSeniorHealth   
2  Symptoms of Glaucoma  Glaucoma can develop in ...  NIHSeniorHealth   
3  Although open-angle glaucoma cannot be cured, ...  NIHSeniorHealth   
4  Glaucoma is a group of diseases that can damag...  NIHSeniorHealth   

  focus_area  
0   Glaucoma  
1   Glaucoma  
2   Glaucoma  
3   Glaucoma  
4   Glaucoma  
Null Value Data:
question       0
answer         5
source         0
focus_area    14
dtype: int64
Number of duplicate rows: 48
Table Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex

,question,answer
0,what is glaucoma ?,glaucoma is a group of diseases that can damag...
1,what causes glaucoma ?,"nearly 2.7 million people have glaucoma, a lea..."
2,what are the symptoms of glaucoma ?,symptoms of glaucoma glaucoma can develop in o...
3,what are the treatments for glaucoma ?,"although open-angle glaucoma cannot be cured, ..."
4,who is at risk for glaucoma? ?,anyone can develop glaucoma. some people are a...


In [5]:
# Define the model name and load the T5 configuration
model_name = "t5-base"
config = T5Config.from_pretrained(model_name)

# Customize the configuration
config.dropout_rate = 0.1  # Set dropout rate to 0.1 for regularization
config.feed_forward_proj = "gelu"  # Use GELU activation for the feed-forward layers

# Load the pre-trained T5 model with the customized configuration
model = T5ForConditionalGeneration.from_pretrained(model_name, config=config)

# Load the tokenizer for the T5 model
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Explicitly resize the token embeddings to match the tokenizer's vocabulary size
model.resize_token_embeddings(len(tokenizer))

# Print a detailed summary of the model architecture
print("\nDetailed Model Summary:")
print("=" * 50)

def summarize_model_by_type(model):
    """
    Summarizes the model by counting the number of layers and parameters for each layer type.
    """
    layer_summary = defaultdict(int)  # Counts the number of layers by type
    param_summary = defaultdict(int)  # Counts the number of parameters by layer type

    for name, module in model.named_modules():
        layer_type = type(module).__name__  # Get the type of the current module
        layer_summary[layer_type] += 1  # Increment the count for this layer type
        param_summary[layer_type] += sum(p.numel() for p in module.parameters())  # Sum parameters

    # Print the summary table
    print(f"{'Layer Type':<30}{'Count':<10}{'Parameters':<15}")
    print("=" * 55)
    for layer_type, count in layer_summary.items():
        print(f"{layer_type:<30}{count:<10}{param_summary[layer_type]:<15,}")

summarize_model_by_type(model)

# Define a preprocessing function for the seq2seq task
def preprocess_function(batch):
    """
    Preprocesses the dataset by tokenizing the inputs and targets.
    """
    # Format the inputs and targets
    inputs = [f"answer the following question: {q}" for q in batch['question']]
    targets = [f"{a}" for a in batch['answer']]

    # Tokenize the inputs
    model_inputs = tokenizer(
        inputs,
        max_length=128,  # Truncate or pad to a maximum length of 128
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

    # Tokenize the targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=64,  # Truncate or pad to a maximum length of 64
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

    # Replace padding token IDs with -100 for the loss function to ignore them
    labels["input_ids"][labels["input_ids"] == tokenizer.pad_token_id] = -100
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Split the dataset into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)

# Convert the pandas DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Preprocess the training and validation datasets
train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=32,  # Process in batches of 32
    remove_columns=train_dataset.column_names,  # Remove original columns
    num_proc=4,  # Use 4 processes for parallel processing
)

val_dataset = val_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=32,  # Process in batches of 32
    remove_columns=val_dataset.column_names,  # Remove original columns
    num_proc=4,  # Use 4 processes for parallel processing
)

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",  # Directory to save the model and results
    evaluation_strategy="epoch",  # Evaluate after each epoch
    save_total_limit=2,  # Keep only the last 2 checkpoints
    learning_rate=5e-4,  # Learning rate
    num_train_epochs=5,  # Number of training epochs
    per_device_train_batch_size=8,  # Batch size for training
    per_device_eval_batch_size=8,  # Batch size for evaluation
    lr_scheduler_type="cosine_with_restarts",  # Learning rate scheduler
    warmup_ratio=0.1,  # Warmup ratio for the scheduler
    weight_decay=0.05,  # Weight decay for regularization
    predict_with_generate=True,  # Generate predictions during evaluation
    fp16=True,  # Use mixed precision for faster training
    logging_dir="./logs",  # Directory for logs
    logging_steps=50,  # Log every 50 steps
    metric_for_best_model="exact_match",  # Use exact match as the primary metric
    greater_is_better=True,  # Higher exact match is better
    report_to="none",  # Disable external reporting
    gradient_accumulation_steps=2,  # Accumulate gradients over 2 steps
    max_grad_norm=0.5,  # Gradient clipping
    optim="adamw_torch_fused",  # Use fused AdamW optimizer
    generation_max_length=64,  # Maximum length for generated text
    generation_num_beams=6,  # Number of beams for beam search
    dataloader_num_workers=4,  # Number of workers for data loading
    group_by_length=True,  # Group sequences by length for efficiency
    remove_unused_columns=True,  # Remove unused columns from the dataset
    label_smoothing_factor=0.1,  # Apply label smoothing
)

# Initialize the data collator for seq2seq tasks
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding='longest',  # Pad sequences to the longest in the batch
    return_tensors="pt",  # Return PyTorch tensors
)

# Define a function to compute evaluation metrics
def compute_metrics(eval_pred, tokenizer):
    """
    Computes exact match, BLEU, and ROUGE-L metrics for evaluation.
    """
    predictions, labels = eval_pred

    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Normalize text for comparison
    decoded_preds = [text.strip().lower() for text in decoded_preds]
    decoded_labels = [text.strip().lower() for text in decoded_labels]

    # Compute exact match
    exact_match = np.mean([p == l for p, l in zip(decoded_preds, decoded_labels)])

    # Load BLEU and ROUGE metrics
    bleu_metric = evaluate.load("bleu")
    rouge_metric = evaluate.load("rouge")

    # Compute BLEU score
    bleu_score = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[label] for label in decoded_labels]
    )["bleu"]

    # Compute ROUGE-L score
    rouge_score = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )["rougeL"]

    return {
        "exact_match": exact_match,
        "BLEU": bleu_score,
        "ROUGE-L": rouge_score,
    }

# Initialize the Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda eval_pred: compute_metrics(eval_pred, tokenizer),
)

# Train the model
trainer.train()

# Save the trained model and tokenizer
trainer.save_model("./t5_chatbot_model")
tokenizer.save_pretrained("./t5_chatbot_tokenizer")

# Save the model's state dictionary
model_path = "./t5_chatbot_model.h5"
torch.save(model.state_dict(), model_path)

# Save the training log history
log_history = trainer.state.log_history

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565



Detailed Model Summary:
Layer Type                    Count     Parameters     
T5ForConditionalGeneration    1         222,882,048    
Embedding                     3         24,653,568     
T5Stack                       2         247,534,848    
ModuleList                    26        396,455,424    
T5Block                       24        198,227,712    
T5LayerSelfAttention          24        56,642,304     
T5Attention                   36        84,935,424     
Linear                        193       222,833,664    
T5LayerNorm                   62        47,616         
Dropout                       86        0              
T5LayerFF                     24        113,264,640    
T5DenseActDense               24        113,246,208    
ReLU                          24        0              
T5LayerCrossAttention         12        28,320,768     


Map (num_proc=4):   0%|          | 0/11778 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2079 [00:00<?, ? examples/s]

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# Initialize lists to store training and evaluation metrics
train_loss = []  # To store training loss values
eval_loss = []  # To store evaluation loss values
eval_bleu = []  # To store BLEU scores during evaluation
eval_exact_match = []  # To store exact match scores during evaluation
eval_rogue = []  # To store ROUGE-L scores during evaluation
steps = []  # To store training step numbers
eval_steps = []  # To store evaluation step numbers

# Extract metrics from the training log history
for log in log_history:
    if "loss" in log:  # Check if training loss is in the log
        train_loss.append(log["loss"])  # Append training loss
        steps.append(log["step"])  # Append the corresponding step number
    if "eval_loss" in log:  # Check if evaluation loss is in the log
        eval_loss.append(log["eval_loss"])  # Append evaluation loss
        eval_steps.append(log["step"])  # Append the corresponding step number
    if "eval_BLEU" in log:  # Check if BLEU score is in the log
        eval_bleu.append(log["eval_BLEU"])  # Append BLEU score
    if "eval_ROUGE-L" in log:  # Check if ROUGE-L score is in the log
        eval_rogue.append(log["eval_ROUGE-L"])  # Append ROUGE-L score
    if "eval_exact_match" in log:  # Check if exact match score is in the log
        eval_exact_match.append(log["eval_exact_match"])  # Append exact match score

# Plot the training and evaluation loss
plt.figure(figsize=(10, 6))
plt.plot(steps, train_loss, label="Training Loss", color="blue", marker="o")  # Plot training loss
plt.plot(steps[:len(eval_loss)], eval_loss, label="Evaluation Loss", color="orange", marker="o")  # Plot evaluation loss
plt.xlabel("Training Steps")  # X-axis label
plt.ylabel("Loss")  # Y-axis label
plt.title("Training vs Evaluation Loss")  # Plot title
plt.legend()  # Show legend
plt.grid(True)  # Add grid for better readability
plt.show()  # Display the plot

# Plot the BLEU score over training steps
plt.figure(figsize=(10, 6))
plt.plot(eval_steps, eval_bleu, label="BLEU", marker="o", linestyle="-", color="green")  # Plot BLEU score
plt.xlabel("Training Steps")  # X-axis label
plt.ylabel("Metric Score")  # Y-axis label
plt.title("BLEU Score Over Training Steps")  # Plot title
plt.legend()  # Show legend
plt.grid(True)  # Add grid for better readability
plt.tight_layout()  # Adjust layout for better spacing
plt.show()  # Display the plot

# Plot the ROUGE-L score over training steps
plt.figure(figsize=(10, 6))
plt.plot(eval_steps, eval_rogue, label="ROUGE-L", marker="o", linestyle="-", color="red")  # Plot ROUGE-L score
plt.xlabel("Training Steps")  # X-axis label
plt.ylabel("Metric Score")  # Y-axis label
plt.title("ROUGE-L Score Over Training Steps")  # Plot title
plt.legend()  # Show legend
plt.grid(True)  # Add grid for better readability
plt.tight_layout()  # Adjust layout for better spacing
plt.show()  # Display the plot

# Plot the exact match score over training steps
plt.figure(figsize=(10, 6))
plt.plot(eval_steps, eval_exact_match, label="Exact Match", marker="o", linestyle="-", color="black")  # Plot exact match score
plt.xlabel("Training Steps")  # X-axis label
plt.ylabel("Metric Score")  # Y-axis label
plt.title("Exact Match Over Training Steps")  # Plot title
plt.legend()  # Show legend
plt.grid(True)  # Add grid for better readability
plt.tight_layout()  # Adjust layout for better spacing
plt.show()  # Display the plot

In [ ]:
# Define paths to the saved model and tokenizer
model_path = "/kaggle/working/t5_chatbot_model"
tokenizer_path = "/kaggle/working/t5_chatbot_tokenizer"

# Load the tokenizer from the saved path
tokenizer = T5Tokenizer.from_pretrained(tokenizer_path)

# Load the model from the saved path
model = T5ForConditionalGeneration.from_pretrained(model_path)

# Set the model to evaluation mode
model.eval()

def generate_response_top_k_top_p(
    question, model, tokenizer, max_length=64, top_k=50, top_p=0.95, temperature=1.0
):
    """
    Generates a response to a given question using Top-K and Top-P sampling.

    Args:
        question (str): The input question to generate a response for.
        model (T5ForConditionalGeneration): The pre-trained T5 model.
        tokenizer (T5Tokenizer): The tokenizer for the T5 model.
        max_length (int): Maximum length of the generated response.
        top_k (int): Number of highest probability tokens to consider for Top-K sampling.
        top_p (float): Cumulative probability threshold for Top-P (nucleus) sampling.
        temperature (float): Controls randomness in sampling (higher = more random).

    Returns:
        str: The generated response.
    """
    # Format the question for the model
    formatted_question = f"Answer the following question: {question}"

    # Tokenize the input question
    inputs = tokenizer(
        formatted_question,
        return_tensors="pt",  # Return PyTorch tensors
        padding=True,  # Pad sequences to the same length
        truncation=True,  # Truncate sequences longer than max_length
        max_length=128,  # Maximum length of the input sequence
    )

    # Generate a response using Top-K and Top-P sampling
    outputs = model.generate(
        input_ids=inputs["input_ids"],  # Input token IDs
        attention_mask=inputs["attention_mask"],  # Attention mask
        max_length=max_length,  # Maximum length of the generated response
        do_sample=True,  # Enable sampling instead of greedy/beam search
        top_k=top_k,  # Top-K sampling: consider the top-k tokens
        top_p=top_p,  # Top-P (nucleus) sampling: consider the smallest set of tokens with cumulative probability >= top_p
        temperature=temperature,  # Adjust randomness (higher values = more random)
        pad_token_id=tokenizer.pad_token_id,  # Token ID for padding
    )

    # Decode the generated response into a human-readable string
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response

# Example usage of the function
question = "What is Alzheimer's?"
response = generate_response_top_k_top_p(question, model, tokenizer)

# Print the question and generated response
print("Question:", question)
print("Response:", response)

In [ ]:
# Define a question to ask the model
question = "I had a surgery which ended up with some failures. What can I do to fix it?"

# Generate a response using the `generate_response_top_k_top_p` function
response = generate_response_top_k_top_p(question, model, tokenizer)

# Print the question and the generated response
print("Question:", question)
print("Response:", response)

In [ ]:
# Define a question about a health concern
question = "I have pain in my back"

# Generate a response using the `generate_response_top_k_top_p` function
response = generate_response_top_k_top_p(question, model, tokenizer)

# Print the question and the generated response
print("Question:", question)
print("Response:", response)

In [ ]:
# Define a question about checking for a serious health condition
question = "How to check if I have cancer"

# Generate a response using the `generate_response_top_k_top_p` function
response = generate_response_top_k_top_p(question, model, tokenizer)

# Print the question and the generated response
print("Question:", question)
print("Response:", response)